# Embeddings — training template

Renders for every `(category, framework)` pair in this family:
`embeddings`.

This is a **template**, not a guide. The pod renders it for one use case with
`notebook.render(category, framework, context)`: `{{ ... }}` placeholders are
substituted from the context, and cells carrying
`metadata.tracebloc.applies_to` are dropped when they do not apply to the pair
being rendered. See `README.md` in this directory for the contract.

**Nothing here calls `start()`.** Start is a button, and there is no Run All.

## This run

| | |
|---|---|
| Use case | {{ use_case }} |
| Dataset | `{{ dataset_id }}` |
| Category | `{{ category }}` |
| Framework | `{{ framework }}` |
| Edges | {{ edge_count }} |
| Records per edge | {{ records_per_edge }} |

In [ ]:
# PENDING A RELEASED SDK. Once the image carries an SDK release with
# environment login, the pod is already authenticated -- it reads its scoped,
# short-lived credential from the environment, so there is no email/password
# prompt here and no token in the notebook.
#
# Until then `User()` PROMPTS interactively, which is wrong for a pod. Note
# that merged is not enough: environment login is on the SDK's `develop`
# (pyproject 1.0.9) but ABSENT from the latest tag v1.0.7, which is what
# `pip install tracebloc` resolves -- so this cell is contingent on a RELEASE,
# not on the change landing. Verified 2026-09-09: v1.0.7 contains no
# `env_login` module and no `TRACEBLOC_TOKEN` path at all.
from tracebloc import User

user = User()

In [ ]:
# Filled in by the model picker. Change the path to point at your own file.
MODEL_PATH = "{{ model_path }}"

user.upload_model(MODEL_PATH)

In [ ]:
# --- Tokenizer -----------------------------------------------------------
# NLP models must ship a tokenizer; there is no fallback. The SDK picks up a
# `<model>_tokenizer.json` sitting next to the model file, so most zoo models
# need nothing here. Pass one explicitly only when yours is named differently:
#
#     user.upload_model(MODEL_PATH, tokenizer="{{ tokenizer_path }}")
#
# For a HuggingFace-hosted tokenizer, set the model file's `tokenizer_id`
# instead. An empty `tokenizer_id` is refused at upload, not at training.

In [ ]:
training = user.link_model_dataset("{{ dataset_id }}")

In [ ]:
# ======================================================================
# Settings — the complete plan for this run, as plain SDK calls.
# Edit a value, then press Start. Nothing here calls start().
# ======================================================================

# --- Experiment ----------------------------------------------------------
training.experiment_name("{{ experiment_name }}")


# --- Federation ----------------------------------------------------------
# cycles = federated rounds; epochs = local epochs per round
# Contrastive training benefits from more rounds of in-batch negatives rather than more local epochs over the same batches.
# https://docs.tracebloc.io/join-use-case/hyperparameters#training-parameters
training.cycles(8)
training.epochs(1)
# Plain FedAvg is safe here because epochs is 1: there is no local drift to
# correct. Raising epochs above 1 means moving to a drift-correcting strategy
# (fedprox, fedadam, fedyogi, fedadagrad) in the same edit.
training.aggregation_strategy("fedavg")

# --- Optimization --------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#1-optimizer
training.optimizer("sgd")
training.learning_rate({"type": "constant", "value": 0.001})
training.seed(0)                              # 0 means no fixed seed

# --- Data ----------------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#dataset-parameters-optional
training.validation_split({{ validation_split }})
training.training_classes({{ training_classes }})

# --- Sequence ------------------------------------------------------------
# Maximum token sequence length. Longer costs quadratically in attention.
# https://docs.tracebloc.io/join-use-case/how-training-works#per-use-case
training.sequence_length({{ sequence_length }})

# --- LoRA ----------------------------------------------------------------
# Off by default. Target modules are derived for you; only the four knobs
# below are yours. LoRA changes what is averaged: adapters, not full weights.
# https://docs.tracebloc.io/join-use-case/hyperparameters#llm-parameters-text-classification
# training.enable_lora(True)
# training.set_lora_parameters(256, 512, 0.05, False)   # r, alpha, dropout, q_lora

# --- Callbacks -----------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#callbacks
training.terminate_on_nan_callback()
# training.early_stop_callback(monitor="val_loss", patience=3)
# training.model_checkpoint_callback(monitor="val_loss", save_best_only=True)
# training.reduce_lr_callback(monitor="val_loss", factor=0.1, patience=2, min_delta=1e-4)

## Start

Press **Start**. It re-links the model and dataset, executes the settings cell
above, and then starts the experiment — in that order, because `start()` is
one-shot and resets the plan. Your remaining team budget is shown beside the
button.

To iterate: change a value above and press Start again.

Prefer to leave? *Download .ipynb* and *Copy as script* both give you the same
settings as plain SDK calls.